In [1]:
from src.misc import *
from src.SPECTRUM import Spectrum
from src.FITSPECTRUM import FitSpectrum
from src.DP import DP
import matplotlib.pyplot as plt
import numpy as np

In [2]:
spectra_data    = fits.open('/Users/hyp0515/data/0715_Spring_BGS_ALL_trimmed.fits')
cigale_data     = fits.open('/Users/hyp0515/data/IronPhysProp_v1.2_extracted.fits')
fastspecfit     = fits.open('/Users/hyp0515/data/0715_Spring_half_BGS_BRIGHT_catalog_fastspecfit.fits')

# Select parent sources who have at least one significant emission line (>5 S/N ratio)

In [3]:
FIT = FitSpectrum()

ALL_SPECTRA = Spectrum(spectra_data, cigale_data, fastspecfit, load_targetID=None)
ALL_SPECTRA = ALL_SPECTRA.subtype_filter(subtype='QSO', exclude=True)
ALL_SPECTRA = ALL_SPECTRA.stack_data()
ALL_SPECTRA = ALL_SPECTRA.shift_to_rest_frame()
print('Total number of spectra:', len(ALL_SPECTRA.targetID))

Total number of spectra: 99812


In [4]:
ALL_SPECTRA = FIT.label_emission_lines(ALL_SPECTRA, 5)
ALL_SPECTRA = FIT.significant_emission_filter(ALL_SPECTRA)
print('Total number of spectra:', len(ALL_SPECTRA.targetID))

Total number of spectra: 58596


## For testing, shrink dataset

In [5]:
ALL_SPECTRA = ALL_SPECTRA.shrink_dataset(10)
print('Number of spectra after shrinking:', len(ALL_SPECTRA.targetID))

Number of spectra after shrinking: 5860


# Fit all sources with both 1-component and 2-component model

In [6]:
DP = DP()
parent_df, candidate_df, sample_df = DP.get_dp_candidate(data_class=ALL_SPECTRA, n_jobs=10)
DP.record_ids(df=parent_df, fname='dp_parent_ids.txt')
DP.record_ids(df=sample_df, fname='dp_sample_ids.txt')

100%|██████████| 5860/5860 [00:57<00:00, 101.51it/s]


In [7]:
PARENT_SPECTRA = Spectrum(spectra_data, cigale_data, fastspecfit, load_targetID=read_ids('dp_parent_ids.txt'))
PARENT_SPECTRA = PARENT_SPECTRA.stack_data()
PARENT_SPECTRA = PARENT_SPECTRA.shift_to_rest_frame()
PARENT_SPECTRA = FIT.label_emission_lines(PARENT_SPECTRA, 5)
DP.get_catalog(data_class=PARENT_SPECTRA, df=parent_df, fname='parent_sample_catalog.fits', n_jobs=10)

100%|██████████| 5860/5860 [00:46<00:00, 127.24it/s]


In [8]:
DP_SPECTRA = Spectrum(spectra_data, cigale_data, fastspecfit, load_targetID=read_ids('dp_sample_ids.txt'))
DP_SPECTRA = DP_SPECTRA.stack_data()
DP_SPECTRA = DP_SPECTRA.shift_to_rest_frame()
DP_SPECTRA = FIT.label_emission_lines(DP_SPECTRA, 3)
DP.get_catalog(data_class=DP_SPECTRA, df=sample_df, fname='dp_sample_catalog.fits', n_jobs=10)

100%|██████████| 1308/1308 [00:07<00:00, 164.72it/s]
